In [ ]:
import os
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict, Any

def load_nuscenes_info(pkl_path: str) -> Dict[str, Any]:
    """
    加载nuScenes数据集的info文件
    
    Args:
        pkl_path: pkl文件路径
    
    Returns:
        包含数据集信息的字典
    """
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    return data

def check_file_exists(file_path: str) -> bool:
    """
    检查文件是否存在
    
    Args:
        file_path: 文件路径
    
    Returns:
        文件是否存在
    """
    return os.path.exists(file_path)

def extract_file_paths_from_nuscenes_info(info_data: Dict[str, Any], data_root: str) -> List[str]:
    """
    从nuScenes info数据中提取所有相关的文件路径
    
    Args:
        info_data: nuScenes info数据
        data_root: 数据根目录
    
    Returns:
        所有需要的文件路径列表
    """
    file_paths = []
    
    # 从infos中提取点云文件路径
    if 'infos' in info_data:
        for info in info_data['infos']:
            # 提取点云文件路径
            lidar_path = info.get('lidar_path', '')
            if lidar_path:
                # 如果路径是相对路径，则添加根目录
                if not lidar_path.startswith('/'):
                    full_path = os.path.join(data_root, lidar_path)
                else:
                    full_path = lidar_path
                file_paths.append(full_path)
            
            # 提取 sweeps 中的点云文件路径
            if 'sweeps' in info:
                for sweep in info['sweeps']:
                    sweep_path = sweep.get('data_path', '')
                    if sweep_path:
                        if not sweep_path.startswith('/'):
                            full_path = os.path.join(data_root, sweep_path)
                        else:
                            full_path = sweep_path
                        file_paths.append(full_path)
            
            # 提取相机图片路径
            if 'cams' in info:
                for cam_name, cam_data in info['cams'].items():
                    cam_path = cam_data.get('data_path', '')
                    if cam_path:
                        if not cam_path.startswith('/'):
                            full_path = os.path.join(data_root, cam_path)
                        else:
                            full_path = cam_path
                        file_paths.append(full_path)
    
    return file_paths

def check_nuscenes_dataset_completeness(pkl_paths: List[str], data_root: str) -> Dict[str, Any]:
    """
    检查nuScenes数据集完整性
    
    Args:
        pkl_paths: info pkl文件路径列表
        data_root: 数据根目录
    
    Returns:
        检查结果字典
    """
    all_file_paths = []
    
    # 从所有pkl文件中收集文件路径
    for pkl_path in pkl_paths:
        print(f"Loading info from {pkl_path}...")
        info_data = load_nuscenes_info(pkl_path)
        file_paths = extract_file_paths_from_nuscenes_info(info_data, data_root)
        all_file_paths.extend(file_paths)
    
    print(f"Total files to check: {len(all_file_paths)}")
    
    # 检查文件是否存在
    missing_files = []
    existing_files = []
    
    for i, file_path in enumerate(all_file_paths):
        if i % 1000 == 0:
            print(f"Checking file {i}/{len(all_file_paths)}...")
        
        if not check_file_exists(file_path):
            missing_files.append(file_path)
        else:
            existing_files.append(file_path)
    
    # 统计结果
    total_files = len(all_file_paths)
    missing_count = len(missing_files)
    existing_count = len(existing_files)
    
    result = {
        'total_files': total_files,
        'existing_files': existing_count,
        'missing_files': missing_count,
        'missing_file_list': missing_files,
        'existing_file_list': existing_files,
        'completeness_rate': existing_count / total_files if total_files > 0 else 0
    }
    
    return result

def analyze_missing_file_types(missing_files: List[str]) -> Dict[str, Any]:
    """
    分析缺失文件的类型分布
    
    Args:
        missing_files: 缺失文件列表
    
    Returns:
        文件类型分析结果
    """
    file_type_counts = {}
    lidar_top_count = 0
    camera_count = 0
    sweeps_count = 0
    samples_count = 0
    
    for file_path in missing_files:
        basename = os.path.basename(file_path)
        dirname = os.path.dirname(file_path)
        
        # 统计文件类型
        if 'LIDAR_TOP' in dirname:
            lidar_top_count += 1
        elif any(cam_type in dirname for cam_type in ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT', 
                                                      'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']):
            camera_count += 1
        
        if 'sweeps/' in dirname:
            sweeps_count += 1
        elif 'samples/' in dirname:
            samples_count += 1
            
        # 按扩展名统计
        ext = os.path.splitext(basename)[1].lower()
        file_type_counts[ext] = file_type_counts.get(ext, 0) + 1
    
    # 按目录路径统计
    dir_counts = {}
    for file_path in missing_files:
        dirname = os.path.dirname(file_path)
        # 只保留最后一级目录名
        last_dir = os.path.basename(dirname)
        dir_counts[last_dir] = dir_counts.get(last_dir, 0) + 1
    
    return {
        'lidar_top_count': lidar_top_count,
        'camera_count': camera_count,
        'sweeps_count': sweeps_count,
        'samples_count': samples_count,
        'file_type_counts': file_type_counts,
        'dir_counts': dir_counts
    }

def print_detailed_report(result: Dict[str, Any]):
    """
    打印详细的检查报告
    """
    print("\n" + "="*60)
    print("NU-SCENES DATASET COMPLETENESS REPORT")
    print("="*60)
    
    print(f"Total files to check: {result['total_files']}")
    print(f"Existing files: {result['existing_files']}")
    print(f"Missing files: {result['missing_files']}")
    print(f"Completeness rate: {result['completeness_rate']:.2%}")
    
    if result['missing_files'] > 0:
        # 分析缺失文件类型
        analysis = analyze_missing_file_types(result['missing_file_list'])
        
        print(f"\nFILE TYPE ANALYSIS:")
        print(f"  LIDAR_TOP files missing: {analysis['lidar_top_count']}")
        print(f"  Camera files missing: {analysis['camera_count']}")
        print(f"  Sweep files missing: {analysis['sweeps_count']}")
        print(f"  Sample files missing: {analysis['samples_count']}")
        
        print(f"\nFILE EXTENSION DISTRIBUTION:")
        for ext, count in analysis['file_type_counts'].items():
            print(f"  {ext}: {count} files")
        
        print(f"\nDIRECTORY DISTRIBUTION:")
        for dir_name, count in sorted(analysis['dir_counts'].items(), key=lambda x: x[1], reverse=True)[:10]:
            print(f"  {dir_name}: {count} files")
        
        print(f"\nALL MISSING FILES BY TYPE:")
        lidar_top_files = [f for f in result['missing_file_list'] if 'LIDAR_TOP' in f]
        camera_files = [f for f in result['missing_file_list'] if any(cam_type in f for cam_type in 
                    ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT'])]
        other_files = [f for f in result['missing_file_list'] if f not in lidar_top_files and f not in camera_files]
        
        print(f"  LIDAR_TOP: {len(lidar_top_files)} files")
        print(f"  Camera: {len(camera_files)} files")
        print(f"  Other: {len(other_files)} files")
        
        print(f"\nAre ALL missing files LIDAR_TOP? {len(lidar_top_files) == len(result['missing_file_list'])}")
        
        print(f"\nFirst 10 missing files:")
        for i, missing_file in enumerate(result['missing_file_list'][:10]):
            file_type = "LIDAR_TOP" if "LIDAR_TOP" in missing_file else (
                "CAMERA" if any(cam_type in missing_file for cam_type in 
                               ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT', 
                                'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']) else "OTHER"
            )
            print(f"  {i+1}. [{file_type}] {missing_file}")
        
        if result['missing_files'] > 10:
            print(f"  ... and {result['missing_files'] - 10} more files")
    else:
        print("\n✅ All files are present! Dataset is complete.")
    
    print("="*60)

def check_specific_missing_file(missing_file_path: str, data_root: str) -> None:
    """
    检查特定缺失文件的详细信息
    """
    print(f"\nChecking specific missing file: {missing_file_path}")
    
    # 检查目录结构
    file_dir = os.path.dirname(missing_file_path)
    parent_dir = os.path.dirname(file_dir)
    
    print(f"File directory exists: {os.path.exists(file_dir)}")
    print(f"Parent directory exists: {os.path.exists(parent_dir)}")
    
    if os.path.exists(parent_dir):
        print(f"Contents of parent directory:")
        try:
            contents = os.listdir(parent_dir)
            for item in contents[:20]:  # 显示前20个文件
                print(f"  - {item}")
            if len(contents) > 20:
                print(f"  ... and {len(contents) - 20} more items")
        except Exception as e:
            print(f"Error listing directory: {e}")

# 示例使用
if __name__ == "__main__":
    # 根据配置文件，设置数据根目录
    data_root = '/home/zhengnanfang/PTv3_proj/UniAD'  # 或者 '/data/nuscenes-mini/' 如果你使用的是mini数据集
    
    # 设置info文件路径
    info_dir = '/home/zhengnanfang/PTv3_proj/UniAD/data/infos'
    pkl_files = [
        os.path.join(info_dir, 'nuscenes_infos_temporal_train.pkl'),
        os.path.join(info_dir, 'nuscenes_infos_temporal_val.pkl')
    ]
    
    # 检查数据集完整性
    result = check_nuscenes_dataset_completeness(pkl_files, data_root)
    
    # 打印报告
    print_detailed_report(result)
    
    # 如果有缺失文件，检查特定文件
    if result['missing_files'] > 0:
        # 检查你遇到的特定错误文件
        error_file = './data/nuscenes/sweeps/LIDAR_TOP/n008-2018-08-28-16-16-48-0400__LIDAR_TOP__1535488460646939.pcd.bin'
        abs_error_file = os.path.abspath(error_file)
        print(f"\nSpecific error file check: {abs_error_file}")
        print(f"File exists: {os.path.exists(abs_error_file)}")
        
        if not os.path.exists(abs_error_file):
            # 尝试检查相对路径
            alt_path = os.path.join(data_root, 'sweeps/LIDAR_TOP/n008-2018-08-28-16-16-48-0400__LIDAR_TOP__1535488460646939.pcd.bin')
            print(f"Alternative path check: {alt_path}")
            print(f"Alternative path exists: {os.path.exists(alt_path)}")

Loading info from /home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_train.pkl...
Loading info from /home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_val.pkl...
Total files to check: 571924
Checking file 0/571924...
Checking file 1000/571924...
Checking file 2000/571924...
Checking file 3000/571924...
Checking file 4000/571924...
Checking file 5000/571924...
Checking file 6000/571924...
Checking file 7000/571924...
Checking file 8000/571924...
Checking file 9000/571924...
Checking file 10000/571924...
Checking file 11000/571924...
Checking file 12000/571924...
Checking file 13000/571924...
Checking file 14000/571924...
Checking file 15000/571924...
Checking file 16000/571924...
Checking file 17000/571924...
Checking file 18000/571924...
Checking file 19000/571924...
Checking file 20000/571924...
Checking file 21000/571924...
Checking file 22000/571924...
Checking file 23000/571924...
Checking file 24000/571924...
Checking file 25000/571924...
Chec

In [1]:
import os
import pickle
from pathlib import Path

def quick_check_missing_file():
    """
    快速检查你遇到的具体缺失文件
    """
    # 检查错误中提到的文件
    error_file = './data/nuscenes/sweeps/LIDAR_TOP/n015-2018-08-03-15-00-36+0800__LIDAR_TOP__1533279663100501.pcd.bin'
    
    print("Checking the specific missing file from error:")
    print(f"Path: {error_file}")
    print(f"Exists: {os.path.exists(error_file)}")
    
    # 检查数据根目录
    data_dirs_to_check = [
        '/data/nuscenes',
        '/data/nuscenes-mini',
        './data/nuscenes',
        '/home/zhengnanfang/PTv3_proj/data/nuscenes'
    ]
    
    print("\nChecking common data root directories:")
    for data_dir in data_dirs_to_check:
        exists = os.path.exists(data_dir)
        print(f"  {data_dir}: {'✅' if exists else '❌'}")
        
        if exists:
            sweeps_dir = os.path.join(data_dir, 'sweeps', 'LIDAR_TOP')
            sweeps_exists = os.path.exists(sweeps_dir)
            print(f"    sweeps/LIDAR_TOP: {'✅' if sweeps_exists else '❌'}")
            
            if sweeps_exists:
                print(f"    Files in sweeps/LIDAR_TOP: {len(os.listdir(sweeps_dir))} files")
    
    # 检查你的配置文件中定义的路径
    print(f"\nBased on your config file:")
    print(f"  data_root = '/data/nuscenes-mini/'")
    print(f"  ann_root = '/home/zhengnanfang/PTv3_proj/CMT/data/nuscenes-mini/'")
    
    config_data_root = '/data/nuscenes-mini/'
    expected_file = os.path.join(config_data_root, 'sweeps/LIDAR_TOP/n015-2018-08-03-15-00-36+0800__LIDAR_TOP__1533279663100501.pcd.bin')
    print(f"  Expected file path: {expected_file}")
    print(f"  File exists: {os.path.exists(expected_file)}")

def check_uniad_infos():
    """
    检查UniAD info文件中的内容
    """
    info_files = [
        '/home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_train.pkl',
        '/home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_val.pkl'
    ]
    
    for info_file in info_files:
        if os.path.exists(info_file):
            print(f"\nLoading {info_file}...")
            with open(info_file, 'rb') as f:
                data = pickle.load(f)
            
            print(f"  Keys in info: {list(data.keys())}")
            
            if 'infos' in data and len(data['infos']) > 0:
                first_info = data['infos'][0]
                print(f"  First sample keys: {list(first_info.keys())}")
                
                if 'sweeps' in first_info and len(first_info['sweeps']) > 0:
                    print(f"  Sample sweep keys: {list(first_info['sweeps'][0].keys())}")
                    print(f"  Sample sweep path: {first_info['sweeps'][0].get('data_path', 'N/A')}")

# 运行检查
quick_check_missing_file()
check_uniad_infos()

Checking the specific missing file from error:
Path: ./data/nuscenes/sweeps/LIDAR_TOP/n015-2018-08-03-15-00-36+0800__LIDAR_TOP__1533279663100501.pcd.bin
Exists: False

Checking common data root directories:
  /data/nuscenes: ✅
    sweeps/LIDAR_TOP: ✅
    Files in sweeps/LIDAR_TOP: 267972 files
  /data/nuscenes-mini: ✅
    sweeps/LIDAR_TOP: ✅
    Files in sweeps/LIDAR_TOP: 3531 files
  ./data/nuscenes: ❌
  /home/zhengnanfang/PTv3_proj/data/nuscenes: ❌

Based on your config file:
  data_root = '/data/nuscenes-mini/'
  ann_root = '/home/zhengnanfang/PTv3_proj/CMT/data/nuscenes-mini/'
  Expected file path: /data/nuscenes-mini/sweeps/LIDAR_TOP/n015-2018-08-03-15-00-36+0800__LIDAR_TOP__1533279663100501.pcd.bin
  File exists: False

Loading /home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_train.pkl...
  Keys in info: ['infos', 'metadata']
  First sample keys: ['lidar_path', 'token', 'prev', 'next', 'can_bus', 'frame_idx', 'sweeps', 'cams', 'scene_token', 'lidar2ego_transl

In [8]:
import os
import pickle
from pathlib import Path
import json

def load_nuscenes_info(pkl_path: str) -> dict:
    """
    加载nuScenes数据集的info文件
    """
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    return data

def extract_all_file_paths(info_data: dict, data_root: str) -> set:
    """
    从nuScenes info数据中提取所有相关的文件路径
    """
    file_paths = set()
    
    if 'infos' in info_data:
        for info in info_data['infos']:
            # 提取点云文件路径
            lidar_path = info.get('lidar_path', '')
            if lidar_path:
                if not lidar_path.startswith('/'):
                    full_path = os.path.join(data_root, lidar_path)
                else:
                    full_path = lidar_path
                file_paths.add(full_path)
            
            # 提取 sweeps 中的点云文件路径
            if 'sweeps' in info:
                for sweep in info['sweeps']:
                    sweep_path = sweep.get('data_path', '')
                    if sweep_path:
                        if not sweep_path.startswith('/'):
                            full_path = os.path.join(data_root, sweep_path)
                        else:
                            full_path = sweep_path
                        file_paths.add(full_path)
            
            # 提取相机图片路径
            if 'cams' in info:
                for cam_name, cam_data in info['cams'].items():
                    cam_path = cam_data.get('data_path', '')
                    if cam_path:
                        if not cam_path.startswith('/'):
                            full_path = os.path.join(data_root, cam_path)
                        else:
                            full_path = cam_path
                        file_paths.add(full_path)
    
    return file_paths

def check_nuscenes_dataset_completeness_extended(pkl_paths: list, data_root: str):
    """
    检查nuScenes数据集完整性，扩展版
    """
    all_file_paths = set()
    
    # 从所有pkl文件中收集文件路径
    for pkl_path in pkl_paths:
        print(f"Loading info from {pkl_path}...")
        info_data = load_nuscenes_info(pkl_path)
        file_paths = extract_all_file_paths(info_data, data_root)
        all_file_paths.update(file_paths)
        print(f"  Found {len(file_paths)} file paths in this info file")
    
    print(f"\nTotal unique files to check: {len(all_file_paths)}")
    
    # 检查文件是否存在
    missing_files = []
    existing_files = []
    
    for i, file_path in enumerate(all_file_paths):
        if i % 10000 == 0:
            print(f"Checking file {i}/{len(all_file_paths)}...")
        
        if not os.path.exists(file_path):
            missing_files.append(file_path)
        else:
            existing_files.append(file_path)
    
    # 统计结果
    total_files = len(all_file_paths)
    missing_count = len(missing_files)
    existing_count = len(existing_files)
    
    result = {
        'total_files': total_files,
        'existing_files': existing_count,
        'missing_files': missing_count,
        'missing_file_list': missing_files,
        'existing_file_list': existing_files,
        'completeness_rate': existing_count / total_files if total_files > 0 else 0
    }
    
    return result

def print_complete_report(result: dict):
    """
    打印完整的检查报告
    """
    print("\n" + "="*80)
    print("NU-SCENES DATASET COMPLETENESS REPORT")
    print("="*80)
    
    print(f"Total files to check: {result['total_files']:,}")
    print(f"Existing files: {result['existing_files']:,}")
    print(f"Missing files: {result['missing_files']:,}")
    print(f"Completeness rate: {result['completeness_rate']:.2%}")
    
    print("\n" + "-"*80)
    print("MISSING FILES LIST:")
    print("-"*80)
    
    if result['missing_files'] > 0:
        print(f"\nAll {result['missing_files']} missing files:")
        for i, missing_file in enumerate(result['missing_file_list'], 1):
            print(f"{i:5d}. {missing_file}")
    else:
        print("\n✅ All files are present! Dataset is complete.")
    
    print("\n" + "-"*80)
    print("SUMMARY BY FILE TYPE:")
    print("-"*80)
    
    # 按文件类型统计
    file_type_counts = {}
    for file_path in result['missing_file_list']:
        file_ext = os.path.splitext(file_path)[1].lower()
        file_type_counts[file_ext] = file_type_counts.get(file_ext, 0) + 1
    
    for file_type, count in sorted(file_type_counts.items()):
        print(f"{file_type:10s}: {count:5d} files")
    
    print("\n" + "-"*80)
    print("SUMMARY BY DIRECTORY:")
    print("-"*80)
    
    # 按目录统计
    dir_counts = {}
    for file_path in result['missing_file_list']:
        directory = os.path.dirname(file_path)
        dir_counts[directory] = dir_counts.get(directory, 0) + 1
    
    # 按缺失数量排序并显示前20个
    sorted_dirs = sorted(dir_counts.items(), key=lambda x: x[1], reverse=True)
    for directory, count in sorted_dirs[:20]:
        print(f"{count:5d} files missing in: {directory}")
    
    if len(sorted_dirs) > 20:
        print(f"... and {len(sorted_dirs) - 20} more directories")
    
    print("="*80)

# 设置路径并执行检查
data_root = '/home/zhengnanfang/PTv3_proj/UniAD'  # 或者 '/data/nuscenes-mini/' 如果你使用的是mini数据集

# 设置info文件路径
info_dir = '/home/zhengnanfang/PTv3_proj/UniAD/data/infos'
pkl_files = [
    os.path.join(info_dir, 'nuscenes_infos_temporal_train.pkl'),
    os.path.join(info_dir, 'nuscenes_infos_temporal_val.pkl')
]

# 检查数据集完整性
result = check_nuscenes_dataset_completeness_extended(pkl_files, data_root)

# 打印完整报告
print_complete_report(result)

Loading info from /home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_train.pkl...
  Found 441792 file paths in this info file
Loading info from /home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_val.pkl...
  Found 94522 file paths in this info file

Total unique files to check: 536314
Checking file 0/536314...
Checking file 10000/536314...
Checking file 20000/536314...
Checking file 30000/536314...
Checking file 40000/536314...
Checking file 50000/536314...
Checking file 60000/536314...
Checking file 70000/536314...
Checking file 80000/536314...
Checking file 90000/536314...
Checking file 100000/536314...
Checking file 110000/536314...
Checking file 120000/536314...
Checking file 130000/536314...
Checking file 140000/536314...
Checking file 150000/536314...
Checking file 160000/536314...
Checking file 170000/536314...
Checking file 180000/536314...
Checking file 190000/536314...
Checking file 200000/536314...
Checking file 210000/536314...
Checking

In [ ]:
import os
import pickle
from pathlib import Path

def load_and_check_nuscenes_completeness_and_scenes():
    """
    一键运行版本：检查nuScenes数据集完整性并找出缺失文件所属场景
    """
    # 设置路径
    info_dir = '/home/zhengnanfang/PTv3_proj/UniAD/data/infos'
    data_root = '/data/nuscenes'  # 根据你的需要调整
    
    pkl_files = [
        os.path.join(info_dir, 'nuscenes_infos_temporal_train.pkl'),
        os.path.join(info_dir, 'nuscenes_infos_temporal_val.pkl')
    ]
    
    all_file_paths = set()
    
    # 收集所有文件路径
    for pkl_file in pkl_files:
        print(f"Processing {pkl_file}...")
        with open(pkl_file, 'rb') as f:
            data = pickle.load(f)
        
        if 'infos' in data:
            for info in data['infos']:
                # 收集lidar文件
                lidar_path = info.get('lidar_path', '')
                if lidar_path:
                    if not lidar_path.startswith('/'):
                        full_path = os.path.join(data_root, lidar_path)
                    else:
                        full_path = lidar_path
                    all_file_paths.add(full_path)
                
                # 收集sweeps文件
                if 'sweeps' in info:
                    for sweep in info['sweeps']:
                        sweep_path = sweep.get('data_path', '')
                        if sweep_path:
                            if not sweep_path.startswith('/'):
                                full_path = os.path.join(data_root, sweep_path)
                            else:
                                full_path = sweep_path
                            all_file_paths.add(full_path)
                
                # 收集cams文件
                if 'cams' in info:
                    for cam_data in info['cams'].values():
                        cam_path = cam_data.get('data_path', '')
                        if cam_path:
                            if not cam_path.startswith('/'):
                                full_path = os.path.join(data_root, cam_path)
                            else:
                                full_path = cam_path
                            all_file_paths.add(full_path)
    
    print(f"Total unique file paths collected: {len(all_file_paths):,}")
    
    # 检查文件是否存在
    missing_files = []
    for i, file_path in enumerate(all_file_paths):
        if i % 10000 == 0:
            print(f"Checking {i}/{len(all_file_paths)} files...")
        
        if not os.path.exists(file_path):
            missing_files.append(file_path)
    
    # 打印结果
    print(f"\nMissing files count: {len(missing_files):,}")
    print(f"Existing files count: {len(all_file_paths) - len(missing_files):,}")
    print(f"Completeness rate: {((len(all_file_paths) - len(missing_files)) / len(all_file_paths) * 100):.2f}%")
    
    print("\nAll missing file paths:")
    for i, missing_file in enumerate(missing_files, 1):
        print(f"{i:5d}. {missing_file}")
    
    # 现在找出缺失文件所属的场景
    print(f"\n" + "="*80)
    print("LOOKING UP SCENES FOR MISSING FILES...")
    print("="*80)
    
    try:
        from nuscenes.nuscenes import NuScenes
        
        # 初始化nuScenes API
        nusc = NuScenes(version='v1.0-trainval', dataroot=data_root, verbose=False)
        
        # 创建文件名到sample_data的映射
        filename_to_sample_data = {}
        for sample_data in nusc.sample_data:
            # 获取文件名（带路径）
            full_filename = sample_data['filename']
            filename_to_sample_data[full_filename] = sample_data
            
            # 也添加基本文件名映射
            base_filename = os.path.basename(full_filename)
            if base_filename not in filename_to_sample_data:
                filename_to_sample_data[base_filename] = sample_data
        
        # 分析缺失文件的场景分布
        scene_file_mapping = {}
        
        for missing_file in missing_files:
            filename = os.path.basename(missing_file)
            sample_data_record = None
            
            # 尝试匹配文件名
            if filename in filename_to_sample_data:
                sample_data_record = filename_to_sample_data[filename]
            else:
                # 尝试匹配完整路径
                for full_path, sd_record in filename_to_sample_data.items():
                    if filename in full_path:
                        sample_data_record = sd_record
                        break
            
            if sample_data_record:
                # 获取sample信息
                sample = nusc.get('sample', sample_data_record['sample_token'])
                scene = nusc.get('scene', sample['scene_token'])
                log = nusc.get('log', scene['log_token'])
                
                scene_info = {
                    'scene_name': scene['name'],
                    'location': log['location'],
                    'log_token': log['token'],
                    'scene_token': scene['token']
                }
                
                if scene['name'] not in scene_file_mapping:
                    scene_file_mapping[scene['name']] = {
                        'scene_info': scene_info,
                        'files': []
                    }
                
                scene_file_mapping[scene['name']]['files'].append(missing_file)
            else:
                # 如果无法找到对应的场景信息，标记为未知
                if 'UNKNOWN_SCENE' not in scene_file_mapping:
                    scene_file_mapping['UNKNOWN_SCENE'] = {
                        'scene_info': {'scene_name': 'UNKNOWN_SCENE', 'location': 'Unknown', 'log_token': 'N/A', 'scene_token': 'N/A'},
                        'files': []
                    }
                scene_file_mapping['UNKNOWN_SCENE']['files'].append(missing_file)
        
        # 打印场景分布报告
        print(f"\nMissing files by scene ({len(scene_file_mapping)} scenes affected):")
        print("-" * 100)
        
        total_missing_by_scene = 0
        for scene_name, scene_data in sorted(scene_file_mapping.items(), 
                                           key=lambda x: len(x[1]['files']), 
                                           reverse=True):
            scene_info = scene_data['scene_info']
            file_count = len(scene_data['files'])
            total_missing_by_scene += file_count
            
            print(f"\nScene: {scene_name}")
            print(f"  Location: {scene_info['location']}")
            print(f"  Files missing: {file_count}")
            print(f"  Log token: {scene_info['log_token']}")
            
            # 显示前5个缺失的文件
            for i, file_path in enumerate(scene_data['files'][:5], 1):
                print(f"    {i}. {os.path.basename(file_path)}")
            if len(scene_data['files']) > 5:
                print(f"    ... and {len(scene_data['files']) - 5} more files")
        
        print(f"\nTotal files counted by scene: {total_missing_by_scene}")
        print(f"Total missing files: {len(missing_files)}")
        
        # 关于为什么相机图像缺失可能不影响纯视觉训练的解释
        print("\n" + "="*80)
        print("EXPLANATION: WHY CAMERA DATA MISSING MAY NOT AFFECT VISUAL-ONLY TRAINING")
        print("="*80)
        
        print("""
1. DATA SPLITTING:
   - nuScenes数据集通常被划分为训练集、验证集和测试集
   - 如果缺失的相机图像是在验证/测试集中的，而训练数据完整，则训练不受影响
   - 但如果训练集中的图像缺失，训练过程会受到影响

2. MULTI-VIEW PROCESSING:
   - 一些模型可能使用多个相机视角（如6个摄像头）
   - 如果只是部分视角缺失，模型可能仍能使用其他视角的数据
   - 但如果是某个场景的所有视角都缺失，则该场景样本无法使用

3. BATCH PROCESSING:
   - 在训练过程中，如果某个batch中的某些样本数据缺失，可能会被跳过
   - 这会导致实际训练的样本数少于预期，但训练过程本身不会中断

4. DATA LOADING STRATEGY:
   - 许多训练代码会包含异常处理，当数据加载失败时会跳过该样本
   - 这样可以保证训练过程的连续性，但会影响数据利用率

5. DIFFERENT DATASET VERSIONS:
   - 你可能使用的是nuScenes的子集（如mini版本）
   - 配置文件中指定的路径可能与实际数据路径不匹配
        """)
        
    except ImportError:
        print("nuScenes python-sdk not installed. Install it with: pip install nuscenes-devkit")
        print("Without the SDK, scene information cannot be retrieved.")
    except Exception as e:
        print(f"Error during scene lookup: {e}")
        print("This might be due to incorrect data path or nuScenes version.")

# 运行完整检查
load_and_check_nuscenes_completeness_and_scenes()

Processing /home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_train.pkl...


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f24e7ee6d30>>
Traceback (most recent call last):
  File "/home/zhengnanfang/anaconda3/envs/uniad_ptv3/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


Processing /home/zhengnanfang/PTv3_proj/UniAD/data/infos/nuscenes_infos_temporal_val.pkl...
